# Data Import

In [1]:
# !pip3 install tableone
import os
import subprocess
import numpy as np
import pandas as pd
from tableone import TableOne

# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'cleaned_data.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
my_dataframe = pd.read_csv(name_of_file_in_bucket)
my_dataframe.head()

Copying gs://fc-secure-094fc0a0-519f-48e9-9547-6ae9fa803d5c/data/cleaned_data.csv...
/ [1 files][ 11.2 MiB/ 11.2 MiB]                                                
Operation completed over 1 objects/11.2 MiB.                                     


[INFO] cleaned_data.csv is successfully downloaded into your working space


,person_id,gender,race,ethnicity,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,...,Type 2 Diabetes,Urinary Tract,Vitamin B Deficiency,Vitamin D Deficiency,Yeast Infection,General Health,General Mental Health,General Physical Health,General Quality,General Social
0,2272845,Unknown/Missing,Unknown/Missing,Unknown/Missing,19,No,No,No,No,No,...,No,No,No,No,No,Very Good,Good,Very Good,Good,Very Good
1,2061175,Unknown/Missing,Unknown/Missing,Unknown/Missing,23,Yes,No,No,Yes,No,...,No,No,No,No,No,Fair,Good,Good,Fair,Good
2,2987154,Unknown/Missing,Unknown/Missing,Unknown/Missing,35,No,No,No,No,Yes,...,No,No,No,No,No,Poor,Good,Poor,Fair,Fair
3,1877627,Unknown/Missing,Unknown/Missing,Unknown/Missing,29,No,No,No,No,Yes,...,No,No,No,No,No,Excellent,Good,Very Good,Very Good,Fair
4,1272732,Other,Unknown/Missing,Unknown/Missing,25,No,No,No,No,No,...,No,No,No,No,No,Fair,Poor,Fair,Fair,Fair


In [2]:
df = my_dataframe
df.shape

(35923, 90)

# Correlation Cleaning

In [3]:
# make variable values numeric
numeric_comor = df.replace(['Yes', 'No'],
                           [1, 0])
numeric_comor.head()

,person_id,gender,race,ethnicity,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,...,Type 2 Diabetes,Urinary Tract,Vitamin B Deficiency,Vitamin D Deficiency,Yeast Infection,General Health,General Mental Health,General Physical Health,General Quality,General Social
0,2272845,Unknown/Missing,Unknown/Missing,Unknown/Missing,19,0,0,0,0,0,...,0,0,0,0,0,Very Good,Good,Very Good,Good,Very Good
1,2061175,Unknown/Missing,Unknown/Missing,Unknown/Missing,23,1,0,0,1,0,...,0,0,0,0,0,Fair,Good,Good,Fair,Good
2,2987154,Unknown/Missing,Unknown/Missing,Unknown/Missing,35,0,0,0,0,1,...,0,0,0,0,0,Poor,Good,Poor,Fair,Fair
3,1877627,Unknown/Missing,Unknown/Missing,Unknown/Missing,29,0,0,0,0,1,...,0,0,0,0,0,Excellent,Good,Very Good,Very Good,Fair
4,1272732,Other,Unknown/Missing,Unknown/Missing,25,0,0,0,0,0,...,0,0,0,0,0,Fair,Poor,Fair,Fair,Fair


In [4]:
# create dummy variables for remaining categorical variables - drop unknown/missing
gender_dum = pd.get_dummies(numeric_comor['gender']).drop('Unknown/Missing', axis = 1)
race_dum = pd.get_dummies(numeric_comor['race']).drop('Unknown/Missing', axis = 1)
eth_dum = pd.get_dummies(numeric_comor['ethnicity']).drop('Unknown/Missing', axis = 1)

In [5]:
numeric_temp = numeric_comor.drop(['person_id', 'gender', 'race', 'ethnicity'], axis = 1)
numeric_dum = pd.concat([numeric_temp, gender_dum, race_dum, eth_dum], axis=1)
numeric_dum.head()

,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Female,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino
0,19,0,0,0,0,0,0,1,0,1,...,False,False,False,False,False,False,False,False,False,False
1,23,1,0,0,1,0,0,1,0,0,...,False,False,False,False,False,False,False,False,False,False
2,35,0,0,0,0,1,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,29,0,0,0,0,1,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,25,0,0,0,0,0,0,0,0,1,...,False,False,True,False,False,False,False,False,False,False


In [6]:
numeric_minus_health = numeric_dum.replace([True, False],
                                           [1, 0])
numeric_minus_health.head()

,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Female,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino
0,19,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,23,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,35,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,29,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,25,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0


In [8]:
# get strongly correlation pairs
def get_high_corr_pairs(dataframe):
    # create correlation table
    corr_table = dataframe.corr()
    high_corr_pairs = []

    # iterate through correlation table
    for row_index in range(0, corr_table.shape[0]):
        for col_index in range(0, corr_table.shape[1]):
            corr_val = round(corr_table.iloc[row_index, col_index], 5)
            abs_corr = abs(corr_val)
            # see if absolute value of correlation > 0.75
            #if abs_corr > 0.75 and abs_corr < 1:
            if abs_corr > 0.50 and abs_corr < 1:
                high_corr_pairs.append([[corr_table.columns[col_index], corr_table.index[row_index]], corr_val])

    string_pairs = []

    # get rid of duplicate pairs (var1_var2_corr = var2_var1_corr)
    for pairs in high_corr_pairs:
        pair_cols = pairs[0]
        pair_val = pairs[1]
        sorted_pair = sorted(pair_cols)
        string_pair = sorted_pair[0] + '_' + sorted_pair[1] + '_' + str(pair_val)
        string_pairs.append(string_pair)

    return(list(set(string_pairs)))

## Correlation: Overall Health

In [9]:
cols = ['General Mental Health', 'General Physical Health', 'General Quality', 'General Social']
ovr_health = numeric_minus_health.drop(cols, axis = 1)

In [10]:
# filter out rows with unknown general health
ovr_health = ovr_health[ovr_health['General Health'] != 'Unknown']
ovr_health = ovr_health.reset_index(drop = True)
ovr_health['General Health'].unique()
print(ovr_health.shape)

(35814, 92)


In [11]:
# change values to numeric
numeric_health = ovr_health.replace(['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
                                    [1, 2.05, 3.58, 4.7, 5])

# move quality variable to end
q_var = numeric_health.pop('General Health')
numeric_health.insert(len(numeric_health.columns), 'General Health', q_var)

numeric = numeric_health.apply(pd.to_numeric)
numeric.head()

,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino,General Health
0,19,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,4.70
1,23,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,2.05
2,35,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.00
3,29,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5.00
4,25,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,2.05


In [12]:
get_high_corr_pairs(numeric)

['Female_Male_-0.8583',
 'Not Hispanic/Latino_White_0.52811',
 'Hispanic/Latino_Not Hispanic/Latino_-0.8627',
 'Anxiety_Depression_0.62684']

## Correlation: General Quality

In [13]:
cols = ['General Mental Health', 'General Physical Health', 'General Health', 'General Social']
qual_health = numeric_minus_health.drop(cols, axis = 1)

# filter out rows with unknown general health
qual_health = qual_health[qual_health['General Quality'] != 'Unknown']
qual_health = qual_health.reset_index(drop = True)
qual_health['General Quality'].unique()
print(qual_health.shape)

# change values to numeric
numeric_health = qual_health.replace(['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
                                     [1, 2.05, 3.58, 4.7, 5])

# move quality variable to end
q_var = numeric_health.pop('General Quality')
numeric_health.insert(len(numeric_health.columns), 'General Quality', q_var)

numeric = numeric_health.apply(pd.to_numeric)
numeric.head()

(35809, 92)


,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino,General Quality
0,19,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,3.58
1,23,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,2.05
2,35,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2.05
3,29,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4.70
4,25,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,2.05


In [14]:
get_high_corr_pairs(numeric)

['Not Hispanic/Latino_White_0.52812',
 'Hispanic/Latino_Not Hispanic/Latino_-0.86289',
 'Anxiety_Depression_0.62621',
 'Female_Male_-0.85829']

## Correlation: General Social

In [15]:
cols = ['General Mental Health', 'General Physical Health', 'General Health', 'General Quality']
soc_health = numeric_minus_health.drop(cols, axis = 1)

# filter out rows with unknown general health
soc_health = soc_health[soc_health['General Social'] != 'Unknown']
soc_health = soc_health.reset_index(drop = True)
soc_health['General Social'].unique()
print(soc_health.shape)

# change values to numeric
numeric_health = soc_health.replace(['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
                                    [1, 2.05, 3.58, 4.7, 5])

# move quality variable to end
q_var = numeric_health.pop('General Social')
numeric_health.insert(len(numeric_health.columns), 'General Social', q_var)

numeric = numeric_health.apply(pd.to_numeric)
numeric.head()

(35777, 92)


,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino,General Social
0,19,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,4.70
1,23,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,3.58
2,35,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2.05
3,29,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2.05
4,25,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,2.05


In [16]:
get_high_corr_pairs(numeric)

['Anxiety_Depression_0.62649',
 'Not Hispanic/Latino_White_0.52792',
 'Female_Male_-0.85818',
 'Hispanic/Latino_Not Hispanic/Latino_-0.86317']

## Correlation: General Mental

In [17]:
cols = ['General Social', 'General Physical Health', 'General Health', 'General Quality']
mental_health = numeric_minus_health.drop(cols, axis = 1)

# filter out rows with unknown general health
mental_health = mental_health[mental_health['General Mental Health'] != 'Unknown']
mental_health = mental_health.reset_index(drop = True)
mental_health['General Mental Health'].unique()
print(mental_health.shape)

# change values to numeric
numeric_health = mental_health.replace(['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
                                       [1, 2.05, 3.58, 4.7, 5])

# move quality variable to end
q_var = numeric_health.pop('General Mental Health')
numeric_health.insert(len(numeric_health.columns), 'General Mental Health', q_var)

numeric = numeric_health.apply(pd.to_numeric)
numeric.head()

(35823, 92)


,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino,General Mental Health
0,19,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,3.58
1,23,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,3.58
2,35,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3.58
3,29,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3.58
4,25,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,1.00


In [18]:
get_high_corr_pairs(numeric)

['Hispanic/Latino_Not Hispanic/Latino_-0.86276',
 'Not Hispanic/Latino_White_0.52802',
 'Anxiety_Depression_0.62672',
 'Female_Male_-0.8584']

## Correlation: General Physical

In [19]:
cols = ['General Social', 'General Mental Health', 'General Health', 'General Quality']
phys_health = numeric_minus_health.drop(cols, axis = 1)

# filter out rows with unknown general health
phys_health = phys_health[phys_health['General Physical Health'] != 'Unknown']
phys_health = phys_health.reset_index(drop = True)
phys_health['General Physical Health'].unique()
print(phys_health.shape)

# change values to numeric
numeric_health = phys_health.replace(['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
                                     [1, 2.05, 3.58, 4.7, 5])

# move quality variable to end
q_var = numeric_health.pop('General Physical Health')
numeric_health.insert(len(numeric_health.columns), 'General Physical Health', q_var)

numeric = numeric_health.apply(pd.to_numeric)
numeric.head()

(35797, 92)


,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Male,Other,Asian,Black,Middle Eastern,More than one,White,Hispanic/Latino,Not Hispanic/Latino,General Physical Health
0,19,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,4.70
1,23,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,3.58
2,35,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.00
3,29,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4.70
4,25,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,2.05


In [20]:
get_high_corr_pairs(numeric)

['Not Hispanic/Latino_White_0.52824',
 'Female_Male_-0.85841',
 'Hispanic/Latino_Not Hispanic/Latino_-0.86291',
 'Anxiety_Depression_0.62713']

In [26]:
numeric.corr()

,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,Anxiety,Asthma,Astigmatism,...,Other,Asian,Black,Middle Eastern,More than one,Pacific Islander,White,Hispanic/Latino,Not Hispanic/Latino,General Physical Health
age_years,1.000000,-0.028216,0.087991,-0.087636,0.037284,0.039675,0.035270,-0.016794,0.022331,0.009219,...,-0.054589,-0.070728,0.034379,-0.013707,-0.047709,0.001136,0.052666,-0.047712,0.039505,-0.053053
ADHD,-0.028216,1.000000,0.076311,0.051316,0.060243,0.072667,0.027190,0.220882,0.080308,0.055033,...,0.058842,-0.039029,-0.038868,0.003321,0.015039,-0.006984,0.068536,-0.055555,0.037243,-0.084615
Acid Reflux,0.087991,0.076311,1.000000,0.048618,0.038349,0.211918,0.159697,0.193930,0.194808,0.066090,...,0.006092,-0.054070,0.006395,-0.009785,-0.010681,0.005133,0.047423,-0.032435,0.023433,-0.212674
Acne,-0.087636,0.051316,0.048618,1.000000,0.005140,0.105356,0.024990,0.070763,0.025405,0.046998,...,0.002717,0.003755,-0.020201,-0.009347,0.014606,-0.003031,0.028495,-0.032235,0.023360,0.020416
Alcohol Or Drug,0.037284,0.060243,0.038349,0.005140,1.000000,0.023362,0.018621,0.107099,0.020290,-0.006050,...,0.022925,-0.022115,-0.005722,-0.009745,-0.009318,0.008684,0.025584,-0.015402,0.010593,-0.042499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Pacific Islander,0.001136,-0.006984,0.005133,-0.003031,0.008684,-0.004763,0.020103,-0.009479,0.005147,-0.011680,...,-0.002959,-0.006542,-0.007392,-0.002487,-0.005033,1.000000,-0.033750,0.000292,0.002550,-0.023771
White,0.052666,0.068536,0.047423,0.028495,0.025584,0.022332,-0.040604,0.136161,0.006450,0.075081,...,0.026544,-0.343408,-0.388041,-0.130537,-0.264215,-0.033750,1.000000,-0.416612,0.528244,0.080704
Hispanic/Latino,-0.047712,-0.055555,-0.032435,-0.032235,-0.015402,-0.021720,0.009450,-0.097102,-0.006114,-0.044576,...,-0.011363,-0.100283,-0.092961,-0.025927,-0.037085,0.000292,-0.416612,1.000000,-0.862913,-0.068427
Not Hispanic/Latino,0.039505,0.037243,0.023433,0.023360,0.010593,0.015353,-0.012624,0.077036,0.001082,0.041124,...,0.019205,0.120223,0.117261,0.034566,0.055908,0.002550,0.528244,-0.862913,1.000000,0.076800


# Cross Tabulations

In [9]:
t_one_data = df.drop('person_id', axis = 1)
t_one_data.head()

,gender,race,ethnicity,age_years,ADHD,Acid Reflux,Acne,Alcohol Or Drug,Allergies,Anemia,...,Type 2 Diabetes,Urinary Tract,Vitamin B Deficiency,Vitamin D Deficiency,Yeast Infection,General Health,General Mental Health,General Physical Health,General Quality,General Social
0,Unknown/Missing,Unknown/Missing,Unknown/Missing,19,No,No,No,No,No,No,...,No,No,No,No,No,Very Good,Good,Very Good,Good,Very Good
1,Unknown/Missing,Unknown/Missing,Unknown/Missing,23,Yes,No,No,Yes,No,No,...,No,No,No,No,No,Fair,Good,Good,Fair,Good
2,Unknown/Missing,Unknown/Missing,Unknown/Missing,35,No,No,No,No,Yes,No,...,No,No,No,No,No,Poor,Good,Poor,Fair,Fair
3,Unknown/Missing,Unknown/Missing,Unknown/Missing,29,No,No,No,No,Yes,No,...,No,No,No,No,No,Excellent,Good,Very Good,Very Good,Fair
4,Other,Unknown/Missing,Unknown/Missing,25,No,No,No,No,No,No,...,No,No,No,No,No,Fair,Poor,Fair,Fair,Fair


In [25]:
# function to create cross tabulation tables 
def create_table_one(dataframe, stratification_variable):
    # make outcomes binary
    dataframe = dataframe.replace(['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],
                                  ['Poor/Fair', 'Poor/Fair', 'Good/VGood/Excellent',
                                   'Good/VGood/Excellent', 'Good/VGood/Excellent'])
    
    # filter out unknown values for stratification variable
    filtered_df = dataframe[dataframe[stratification_variable] != 'Unknown']
    
    # set category order for stratification variable
    filtered_df[stratification_variable] = pd.Categorical(values = filtered_df[stratification_variable],
                                                          categories = ['Poor/Fair', 'Good/VGood/Excellent'],
                                                          ordered = True)
    
    # set category order for other outcomes
    outcome_vars = ['General Mental Health', 'General Physical Health',
                    'General Quality', 'General Social', 'General Health']
    
    # remove stratificaton variable from array
    while(stratification_variable in outcome_vars):
        outcome_vars.remove(stratification_variable)
        
    # iterate through and change order for each
    for other_outcome in outcome_vars:
        filtered_df[other_outcome] = pd.Categorical(values = filtered_df[other_outcome],
                                                    categories = ['Unknown', 'Poor/Fair', 'Good/VGood/Excellent'],
                                                    ordered = True)
    
    # columns to summarize
    summ_cols = filtered_df.columns.to_list()
    summ_cols.remove(stratification_variable)

    # columns containing categorical variables
    cat_cols = filtered_df.columns.to_list()
    cat_cols.remove(stratification_variable)
    cat_cols.remove('age_years')

    # stratification variable
    group_var = [stratification_variable]
    
    # create
    res = TableOne(filtered_df, columns = summ_cols, categorical = cat_cols, groupby = group_var)
    return(res)

In [28]:
test_t1 = create_table_one(t_one_data, 'General Quality')
test_t1

/tmp/ipykernel_116/3472986837.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df[stratification_variable] = pd.Categorical(values = filtered_df[stratification_variable],
/tmp/ipykernel_116/3472986837.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df[other_outcome] = pd.Categorical(values = filtered_df[other_outcome],
/tmp/ipykernel_116/3472986837.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

Grouped by General Quality                                                
                                                                                  Missing       Overall    Poor/Fair Good/VGood/Excellent
n                                                                                                 35809         3774                32035
gender, n (%)                             Female                                        0  24735 (69.1)  2487 (65.9)         22248 (69.4)
                                          Male                                              8881 (24.8)   850 (22.5)          8031 (25.1)
                                          Other                                               479 (1.3)    139 (3.7)            340 (1.1)
                                          Unknown/Missing                                    1714 (4.8)    298 (7.9)           1416 (4.4)
race, n (%)                               Asian                                         0    2257 (6.3)    214 (5.7)           2043 (6.4)
                                          Black                                              2802 (7.8)   453 (12.0)           2349 (7.3)
                                          Middle Eastern                                      343 (1.0)     30 (0.8)            313 (1.0)
                                          More than one                                      1357 (3.8)    181 (4.8)           1176 (3.7)
                                          Unknown/Missing                                   6163 (17.2)   791 (21.0)          5372 (16.8)
                                          White                                            22887 (63.9)  2105 (55.8)         20782 (64.9)
ethnicity, n (%)                          Hispanic/Latino                               0   6079 (17.0)   729 (19.3)          5350 (16.7)
                                          Not Hispanic/Latino                              28094 (78.5)  2801 (74.2)         25293 (79.0)
                                          Unknown/Missing                                    1636 (4.6)    244 (6.5)           1392 (4.3)
age_years, mean (SD)                                                                    0    30.7 (5.8)   31.2 (6.0)           30.6 (5.8)
ADHD, n (%)                               No                                            0  33283 (92.9)  3249 (86.1)         30034 (93.8)
                                          Yes                                                2526 (7.1)   525 (13.9)           2001 (6.2)
Acid Reflux, n (%)                        No                                            0  31846 (88.9)  2846 (75.4)         29000 (90.5)
                                          Yes                                               3963 (11.1)   928 (24.6)           3035 (9.5)
Acne, n (%)                               No                                            0  33127 (92.5)  3483 (92.3)         29644 (92.5)
                                          Yes                                                2682 (7.5)    291 (7.7)           2391 (7.5)
Alcohol Or Drug, n (%)                    No                                            0  35459 (99.0)  3677 (97.4)         31782 (99.2)
                                          Yes                                                 350 (1.0)     97 (2.6)            253 (0.8)
Allergies, n (%)                          No                                            0  30254 (84.5)  2905 (77.0)         27349 (85.4)
                                          Yes                                               5555 (15.5)   869 (23.0)          4686 (14.6)
Anemia, n (%)                             No                                            0  33474 (93.5)  3257 (86.3)         30217 (94.3)
                                          Yes                                                2335 (6.5)   517 (13.7)           1818 (5.7)
Anxiety, n (%)                            No                 

In [29]:
# check value counts
t_one_data['General Quality'].value_counts()

General Quality
Very Good    14850
Good          9966
Excellent     7219
Fair          3240
Poor           534
Unknown        114
Name: count, dtype: int64

In [30]:
# convert table to dataframe
test_html = test_t1.to_html()
test_df = pd.read_html(test_html)[0]

# rename columns
test_df.columns = test_df.columns.droplevel(0)
test_df = test_df.rename(columns = {'Unnamed: 0_level_1':'Variable',
                                    'Unnamed: 1_level_1':'Option'})

test_df

,Variable,Option,Missing,Overall,Poor/Fair,Good/VGood/Excellent
0,n,NaN,NaN,35809,3774,32035
1,"gender, n (%)",Female,0.0,24735 (69.1),2487 (65.9),22248 (69.4)
2,"gender, n (%)",Male,NaN,8881 (24.8),850 (22.5),8031 (25.1)
3,"gender, n (%)",Other,NaN,479 (1.3),139 (3.7),340 (1.1)
4,"gender, n (%)",Unknown/Missing,NaN,1714 (4.8),298 (7.9),1416 (4.4)
...,...,...,...,...,...,...
182,"General Physical Health, n (%)",Poor/Fair,NaN,7486 (20.9),2911 (77.1),4575 (14.3)
183,"General Physical Health, n (%)",Good/VGood/Excellent,NaN,28225 (78.8),838 (22.2),27387 (85.5)
184,"General Social, n (%)",Unknown,0.0,125 (0.3),27 (0.7),98 (0.3)
185,"General Social, n (%)",Poor/Fair,NaN,3402 (9.5),1737 (46.0),1665 (5.2)


In [31]:
# save table one dataframe as csv to local
# from workspace -> about -> browse files in google cloud platform -> folder -> data




# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = test_df   

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'test_table_one.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr


b'Copying file://./test_table_one.csv [Content-Type=text/csv]...\n/ [0 files][    0.0 B/ 11.8 KiB]                                                \r/ [1 files][ 11.8 KiB/ 11.8 KiB]                                                \r\nOperation completed over 1 objects/11.8 KiB.                                     \n'